In [6]:
# Packages Used
import streamlit as st
import numpy as np
import base64
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from openai import OpenAI
from PIL import Image
import io

In [7]:
# ------------------------
# Load trained CNN model
# ------------------------
MODEL_PATH = "xray_cnn_model.h5"
model = load_model(MODEL_PATH)

In [8]:
# ------------------------
# OpenAI client
# ------------------------
client = OpenAI(api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")  

# ------------------------
# Prediction function
# ------------------------
def predict_xray(img_path, model):
    img = image.load_img(img_path, target_size=(224, 224), color_mode="grayscale")
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]
    if prediction > 0.5:
        result = "Pneumonia Detected"
    else:
        result = "Normal"
    return result


In [9]:
# ------------------------
# GPT Detailed Report
# ------------------------
def get_detailed_report(image_path, result):

    img = Image.open(image_path)          # already grayscale
    img = img.resize((128, 128))          # resize

    buffered = io.BytesIO()
    img.save(buffered, format="JPEG", quality=50)  # compress
    img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")  # convert to base64

    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "You are a medical assistant. Provide detailed, human-friendly explanations of chest X-ray predictions."
        },
        {
            "role": "user",
            "content": f"The CNN model predicted: {result}. Here is the chest X-ray image (base64): {img_b64}. Please provide a detailed medical-style report and next steps."
        }
    ],
    temperature=0.3,
    max_tokens=700
)

    return response.choices[0].message.content

    

In [10]:
# ------------------------
# Streamlit UI
# ------------------------
st.title("Chest X-Ray Pneumonia Detector + Report")
st.markdown(
    """
    <div style="text-align:center; padding: 15px;">
        <p style="font-size:26px; font-weight:bold;">
            This report is for quick medical assistance only.
        </p>
        <p style="font-size:28px; font-weight:bold;">
            The final diagnosis should always be confirmed by a doctor.
        </p>
    </div>
    """,
    unsafe_allow_html=True
)
uploaded_file = st.file_uploader("Upload a Chest X-ray Image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Show uploaded image
    st.image(uploaded_file, caption="Uploaded Chest X-Ray", width='stretch')

    # Save temp file for model prediction
    with open("temp_xray.jpg", "wb") as f:
        f.write(uploaded_file.getbuffer())

    # CNN Prediction
    result = predict_xray("temp_xray.jpg", model)
    st.write(f"### ✅ CNN Model Prediction: **{result}**")

    # Get GPT Detailed Report
    with st.spinner("🔍 Generating detailed medical report..."):
        report = get_detailed_report("temp_xray.jpg", result)

    st.subheader("📋 Detailed Report")
    st.write(report)


2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-24 18:28:29.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar